In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.linalg import expm

# Part 1

In [ ]:
def generate_augmented_trajectory_minimal(x0, N=100, tf=200.0, m=100.0, F_thrust=5.0):
    """
    Generate spacecraft rendezvous trajectory with minimal convex hull representation of actuator set.
    Uses Zero-Order Hold (ZOH) discretization for dynamics.
    Returns solver time as well.
    """
    u_max = F_thrust / m
    Delta_t = tf / N

    # Orbital dynamics
    mu = 3.986e14
    R = 7102.8e3
    n = np.sqrt(mu / R**3)
    A = np.array([
        [0,0,0,1,0,0],
        [0,0,0,0,1,0],
        [0,0,0,0,0,1],
        [3*n**2,0,0,0,2*n,0],
        [0,0,0,-2*n,0,0],
        [0,0,-n**2,0,0,0]
    ])
    B = np.array([
        [0,0,0],
        [0,0,0],
        [0,0,0],
        [1,0,0],
        [0,1,0],
        [0,0,1]
    ])

    # --- ZOH discretization ---
    nx = A.shape[0]
    nu = B.shape[1]
    M = np.zeros((nx+nu, nx+nu))
    M[:nx, :nx] = A
    M[:nx, nx:] = B
    Md = expm(M * Delta_t)
    A_d = Md[:nx, :nx]
    B_d = Md[:nx, nx:]

    # Minimal extreme points (linearly independent)
    extreme_points = np.array([
        [ u_max,  0, 0, -u_max,  0, 0,  u_max/3, -u_max/3],
        [ 0,  u_max, 0,  0, -u_max, 0,  u_max/3, -u_max/3],
        [ 0,  0, u_max, 0, 0, -u_max, u_max/3, -u_max/3]
    ])  # shape (3, 8)
    num_extremes = extreme_points.shape[1]

    # Tight L1 bound
    u_bar = np.max(np.sum(np.abs(extreme_points), axis=0))

    # Decision variables
    x = cp.Variable((6, N+1))
    x_c = cp.Variable((1, N+1))
    alpha = cp.Variable((num_extremes, N))
    nu = cp.Variable((1, N))

    constraints = [x[:,0] == x0, x_c[:,0] == 0]

    for k in range(N):
        u_k = extreme_points @ alpha[:,k]
        constraints += [
            x[:,k+1] == A_d @ x[:,k] + B_d @ u_k,
            x_c[:,k+1] == x_c[:,k] + Delta_t * nu[:,k],
            cp.sum(alpha[:,k]) == 1,
            alpha[:,k] >= 0,
            nu[:,k] >= 0,
            nu[:,k] >= cp.norm1(u_k),
            nu[:,k] <= u_bar
        ]

    xf = np.zeros(6)
    constraints += [x[:,N] == xf]

    objective = cp.Minimize(x_c[:,N])
    prob = cp.Problem(objective, constraints)

    # Solve without verbose output
    prob.solve(solver=cp.ECOS, verbose=False)

    x_traj = x.value
    x_c_traj = x_c.value
    u_traj = extreme_points @ alpha.value
    nu_traj = nu.value
    solver_time = prob.solver_stats.solve_time  # Solver time in seconds

    return x_traj, x_c_traj, u_traj, nu_traj, prob.value, solver_time


In [ ]:
def plot_cp_trajectory_results(cp_results, tf=200.0, u_max=5/100, save_plots=False, show_title=False):
   
    # Font settings
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman"],
        "mathtext.rm": "Times New Roman",
        'text.usetex': True,
        "axes.labelsize": 9,
        "axes.titlesize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7
    })

    # Unpack
    x_cp, u_cp, _ = cp_results
    N_cp = x_cp.shape[1] - 1
    time_cp = np.linspace(0, tf, N_cp + 1)

    # -----------------------
    # Build discrete input set
    # -----------------------
    discrete_points = []
    for i in range(3):
        for s in (+1, -1):
            v = np.zeros(3); v[i] = s * u_max
            discrete_points.append(v)
    for i in range(3):
        for j in range(i+1, 3):
            for s in (+1, -1):
                v = np.zeros(3); v[i] = s * u_max/2; v[j] = s * u_max/2
                discrete_points.append(v)
    for s in (+1, -1):
        v = np.ones(3) * s * u_max/3
        discrete_points.append(v)
    discrete_points = np.array(discrete_points)  # (P,3)

    # -----------------------
    # Compute average distance for single trajectory
    # -----------------------
    u_traj_array = np.array(u_cp).T  # shape (N,3) if u_cp was (3,N)
    dists = []
    for u_t in u_traj_array:
        diffs = u_t[None, :] - discrete_points  # (P,3)
        dist = np.linalg.norm(diffs, axis=1)   # (P,)
        dists.append(np.min(dist))             # closest discrete point
    avg_distance = np.mean(dists)
    print(f"Average distance to discrete set: {avg_distance:.6f}")
    print(f"Average distance to discrete set: {avg_distance/u_max:.6f}")

    # ------------------------
    # Plot 1: Spacecraft States (6 subplots)
    # ------------------------
    subplot_height = 3.5 / 5  # inches per subplot
    fig_height = 6 * subplot_height
    fig_width = 3.5

    fig, axs = plt.subplots(6, 1, figsize=(fig_width, fig_height), sharex=True)

    state_symbols = ['r_x', 'r_y', 'r_z', 'v_x', 'v_y', 'v_z']
    state_units = [r'[\mathrm{m}]', r'[\mathrm{m}]', r'[\mathrm{m}]',
                   r'[\mathrm{m}/\mathrm{s}]', r'[\mathrm{m}/\mathrm{s}]', r'[\mathrm{m}/\mathrm{s}]']

    for i in range(6):
        axs[i].plot(time_cp, x_cp[i, :], linewidth=1.5, color=f'C{i}')
        axs[i].axhline(0, color='k', linestyle='--', linewidth=0.8)
        label = '$' + state_symbols[i] + r'\;' + state_units[i] + '$'
        axs[i].set_ylabel(label, fontsize=9)
        axs[i].grid(True)
        axs[i].set_xlim([0, tf])
        if show_title and i == 0:
            axs[i].set_title('Spacecraft States')

    axs[-1].set_xlabel('Time [s]', fontsize=9)
    plt.subplots_adjust(hspace=0.25)
    plt.tight_layout(pad=1.0)
    if save_plots:
        plt.savefig("spacecraft_states_cp.pdf", bbox_inches='tight')
    plt.show()

    # ------------------------
    # Plot 2: Control Inputs (3 subplots)
    # ------------------------
    fig, axes = plt.subplots(3, 1, figsize=(3.5, (3.5/4)*3), sharex=True)
    ctrl_symbols = ['u_x', 'u_y', 'u_z']
    ctrl_units   = r'[\mathrm{m}/\mathrm{s}^2]'

    for i, ax in enumerate(axes):
        ax.step(time_cp[:-1], u_cp[i, :], where='post', linewidth=1.5, color=f'C{i}')
        ax.axhline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
        for factor in [1, 0.5, 1/3]:
            ax.axhline(factor*u_max, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
            ax.axhline(-factor*u_max, color='k', linestyle='--', linewidth=0.8, alpha=0.5)

        label = '$' + ctrl_symbols[i] + r'\;' + ctrl_units + '$'
        ax.set_ylabel(label, fontsize=9)
        ax.grid(True)

    axes[-1].set_xlabel('Time [s]', fontsize=9)
    for ax in axes:
        ax.set_xlim([0, tf])

    plt.subplots_adjust(hspace=0.25)
    plt.tight_layout(pad=1.0)
    if save_plots:
        plt.savefig("control_inputs_cp.pdf", bbox_inches='tight')
    plt.show()

    return None


In [ ]:
x0 = np.array([-100.0, -500.0, -100.0, 0, 0, 0])
t_f = 240.0

# Convexified solution
aug_cp_results = generate_augmented_trajectory_minimal(x0, N=1000, tf=t_f)
x_aug, x_c_aug, u_aug, nu_aug, cost_aug, solv_time = aug_cp_results
cp_results = x_aug, u_aug, cost_aug

print(f"Convexified Augmened (CP) trajectory cost: {cost_aug:.4f}")
print(f"Convexified Augmened (CP) Solving time: {solv_time:.4f}")

# Plot both together
plot_cp_trajectory_results(cp_results, tf=t_f, save_plots=False)


# Part 2

In [ ]:
def grid_size_test_feasible_set(num_IC=10, x0_bounds=None, N_0=100, N_f=1000, N_step=10, tf=300.0, max_attempts=5000):
    """
    Perform a grid size test using a set of feasible initial conditions.

    Args:
        num_IC: number of feasible initial conditions to test
        x0_bounds: dict with 'pos' and 'vel' bounds for sampling ICs
        N_0, N_f, N_step: grid sizes
        tf: final time
        max_attempts: maximum attempts to generate feasible ICs

    Returns:
        results: dict with keys:
            'N_list': list of grid sizes
            'solver_times': array (num_IC, num_N)
            'u_all': list of lists of controls per IC per N
            'ICs': array of feasible initial conditions
    """

    if x0_bounds is None:
        x0_bounds = {
            'pos': [-500.0, 500.0],
            'vel': [-5.0, 5.0]
        }

    # -----------------------
    # Generate feasible ICs
    # -----------------------
    ICs = []
    attempts = 0
    while len(ICs) < num_IC and attempts < max_attempts:
        attempts += 1
        x0_candidate = np.zeros(6)
        x0_candidate[:3] = np.random.uniform(x0_bounds['pos'][0], x0_bounds['pos'][1], 3)
        x0_candidate[3:] = np.random.uniform(x0_bounds['vel'][0], x0_bounds['vel'][1], 3)

        try:
            # Try generating trajectory with initial grid size
            aug_cp_results = generate_augmented_trajectory_minimal(x0_candidate, N=N_0, tf=tf)
            ICs.append(x0_candidate)
            print(f"Feasible IC {len(ICs)} found after {attempts} attempts")
        except Exception:
            continue

    if len(ICs) < num_IC:
        raise RuntimeError("Could not find enough feasible initial conditions.")

    # -----------------------
    # Grid size test
    # -----------------------
    N_list = list(range(N_0, N_f + 1, N_step))
    solver_times = np.zeros((num_IC, len(N_list)))
    u_all = [[None for _ in N_list] for _ in range(num_IC)]

    for ic_idx, x0 in enumerate(ICs):
        print(f"\nTesting IC {ic_idx+1}/{num_IC}")
        for n_idx, N in enumerate(N_list):
            try:
                aug_cp_results = generate_augmented_trajectory_minimal(x0, N=N, tf=tf)
                x_aug, x_c_aug, u_aug, nu_aug, cost_aug, solve_time = aug_cp_results
                solver_times[ic_idx, n_idx] = solve_time
                u_all[ic_idx][n_idx] = u_aug
            except Exception as e:
                print(f"N={N} failed for IC {ic_idx+1}: {e}")
                solver_times[ic_idx, n_idx] = np.nan
                u_all[ic_idx][n_idx] = None

    results = {
        'N_list': N_list,
        'solver_times': solver_times,
        'u_all': u_all,
        'ICs': np.array(ICs)
    }

    return results


In [ ]:
def plot_grid_size_tradeoff_mean(results, u_max=5/100, save_plots=False, discreteness_threshold=0.01):
    """
    Plot solver time vs grid size and averaged distance to discrete set vs grid size using multiple ICs.
    Uses mean and shaded standard deviation instead of boxplots.

    Args:
        results: dict returned by `grid_size_test_feasible_set`
        u_max: maximum actuator value
        save_plots: whether to save the figure
        discreteness_threshold: threshold to suggest a grid size
    """

    # -----------------------
    # Font settings (applied locally here)
    # -----------------------
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman"],
        "mathtext.rm": "Times New Roman",
        'text.usetex': True,
        "axes.labelsize": 9,
        "axes.titlesize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7
    })

    N_list = np.array(results['N_list'])
    solver_times = np.array(results['solver_times'])
    u_all = results['u_all']
    num_IC = solver_times.shape[0]

    # -----------------------
    # Build discrete input set
    # -----------------------
    discrete_points = []
    for i in range(3):
        for s in (+1, -1):
            v = np.zeros(3); v[i] = s * u_max
            discrete_points.append(v)
    for i in range(3):
        for j in range(i+1, 3):
            for s in (+1, -1):
                v = np.zeros(3); v[i] = s * u_max/2; v[j] = s * u_max/2
                discrete_points.append(v)
    for s in (+1, -1):
        v = np.ones(3) * s * u_max/3
        discrete_points.append(v)
    discrete_points = np.array(discrete_points)  # (P,3)

    # -----------------------
    # Compute averaged distance
    # -----------------------
    averaged_distances = np.zeros_like(solver_times, dtype=float)
    for ic_idx in range(num_IC):
        for n_idx, u_traj in enumerate(u_all[ic_idx]):
            if u_traj is None:
                averaged_distances[ic_idx, n_idx] = np.nan
                continue

            u_traj_array = np.array(u_traj)
            if u_traj_array.ndim == 2:
                u_traj_array = u_traj_array[None, :, :]  # add trial dimension

            T, _, N = u_traj_array.shape
            dist = np.zeros((T, N))
            for t_idx in range(N):
                u_t = u_traj_array[:, :, t_idx]
                diffs = u_t[:, None, :] - discrete_points[None, :, :]
                dists = np.linalg.norm(diffs, axis=2)
                dist[:, t_idx] = np.min(dists, axis=1)

            averaged_distances[ic_idx, n_idx] = dist.mean()
    
    # -----------------------
    # Compute mean and std
    # -----------------------
    solver_mean = np.nanmean(solver_times, axis=0)
    solver_median = np.nanmedian(solver_times, axis=0)
    solver_std = np.nanstd(solver_times, axis=0)
    distance_mean = np.nanmean(averaged_distances, axis=0)
    distance_median = np.nanmedian(averaged_distances, axis=0)
    distance_std = np.nanstd(averaged_distances, axis=0)
    p10 = np.nanpercentile(averaged_distances, 10, axis=0)
    p90 = np.nanpercentile(averaged_distances, 90, axis=0)


    # -----------------------
    # Plot mean
    # -----------------------
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.5, 2.8), sharex=True)

    # Solver time
    ax1.plot(N_list, solver_mean, color='blue', marker='o', label='Mean')
    # ax1.fill_between(N_list, solver_mean - solver_std, solver_mean + solver_std,
    #                 color='lightblue', alpha=0.6, label='±1 std')
    ax1.set_ylabel('Solver time [s]')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()
    ax1.set_xlim(min(N_list), max(N_list))

    # Averaged distance
    ax2.plot(N_list, distance_mean, color='darkred', marker='o', label='Mean')
    # scale = 1
    # ax2.fill_between(N_list, distance_mean - scale*distance_std,
    #                 distance_mean + scale*distance_std,
    #                 color='lightcoral', alpha=0.6, label='Variability')
    ax2.set_xlabel('Grid size [N]')
    ax2.set_ylabel(r'$\bar{d}(u)$')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend(loc="upper right")
    ax2.set_xlim(min(N_list), max(N_list))

    fig.tight_layout()

    if save_plots:
        plt.savefig("grid_size_tradeoff_mean.pdf", bbox_inches='tight')
    plt.show()

    # -----------------------
    # Plot median
    # -----------------------
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.5, 2.8), sharex=True)

    # Solver time
    ax1.plot(N_list, solver_median, color='blue', marker='o', label='Median')
    # ax1.fill_between(N_list, solver_mean - solver_std, solver_mean + solver_std,
    #                 color='lightblue', alpha=0.6, label='±1 std')
    ax1.set_ylabel('Solver time [s]')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()
    ax1.set_xlim(min(N_list), max(N_list))

    # Averaged distance
    ax2.plot(N_list, distance_median, color='darkred', marker='o', label='Median')
    # scale = 1
    # ax2.fill_between(N_list, distance_mean - scale*distance_std,
    #                 distance_mean + scale*distance_std,
    #                 color='lightcoral', alpha=0.6, label='Variability')
    ax2.set_xlabel('Grid size [N]')
    ax2.set_ylabel(r'$\bar{d}(u)$')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend(loc="upper right")
    ax2.set_xlim(min(N_list), max(N_list))

    fig.tight_layout()

    if save_plots:
        plt.savefig("grid_size_tradeoff_median.pdf", bbox_inches='tight')
    plt.show()


In [ ]:
x0_bounds = {
    'pos': [-500.0, 500.0],
    'vel': [-5.0, 5.0]
}
t_f = 300.0

num_ICs = 10  # number of initial conditions to test

results = grid_size_test_feasible_set(
    num_IC=num_ICs,
    x0_bounds=x0_bounds,
    N_0=100,
    N_f=1000,
    N_step=100,
    tf=t_f
)

plot_grid_size_tradeoff_mean(
    results,
    u_max=5/100,
    save_plots=True,
    discreteness_threshold=0.01
)


# Part 3

In [ ]:
def monte_carlo_cp(num_samples=100, N=300, tf=300.0, bounds=None, adjust_tf=False, tf_increment=50.0, max_tf=1000.0):
    """
    Perform Monte Carlo simulations for the convexified trajectory optimization.

    Args:
        num_samples: number of Monte Carlo trials
        N: number of time steps
        tf: initial final time [s]
        bounds: dict with keys 'pos' and 'vel', each giving [min, max] range for initial conditions
        save_plots: whether to save histogram plots of computation times
        adjust_tf: if True, automatically increase tf for infeasible initial conditions
        tf_increment: time increment for infeasible cases
        max_tf: maximum allowed final time

    Returns:
        costs: list of terminal costs x_c(tf) for each trial
        solver_times: list of solver times for each trial [s]
        u_all: list of control trajectories for envelope plotting
    """
    import numpy as np
    import matplotlib.pyplot as plt

    if bounds is None:
        bounds = {
            'pos': [-500.0, 500.0],  # x, y, z positions
            'vel': [-5.0, 5.0]       # vx, vy, vz velocities
        }

    costs = []
    solver_times = []
    u_all = []

    trial = 0
    attempted = 0  # Count all attempted initial conditions

    while trial < num_samples:
        attempted += 1

        # Sample initial conditions uniformly within bounds
        x0 = np.zeros(6)
        x0[:3] = np.random.uniform(bounds['pos'][0], bounds['pos'][1], 3)
        x0[3:] = np.random.uniform(bounds['vel'][0], bounds['vel'][1], 3)

        current_tf = tf
        feasible = False

        while not feasible:
            try:
                # Call updated function that returns solver time
                aug_cp_results = generate_augmented_trajectory_minimal(x0, N=N, tf=current_tf)
                x_aug, x_c_aug, u_aug, nu_aug, cost_aug, solve_time = aug_cp_results
                feasible = True
            except Exception:
                feasible = False
                if adjust_tf:
                    current_tf += tf_increment
                    if current_tf > max_tf:
                        print(f"Initial condition infeasible, skipping attempt {attempted}.")
                        break
                else:
                    # Skip this initial condition and resample
                    break

        if feasible:
            costs.append(cost_aug)
            solver_times.append(solve_time)
            u_all.append(u_aug)
            trial += 1
            print(f'Trial {trial}/{num_samples} | Attempted: {attempted} | Feasible: {trial} | '
                  f'Terminal cost: {cost_aug:.4f} | Solver time: {solve_time:.4f} s | tf={current_tf:.1f}')

    return solver_times, u_all

In [ ]:
def discreteness_report(u_all, solver_times, u_max=5/100, eps_discrete=0.05, save_plots=False):
    
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman"],
        "mathtext.rm": "Times New Roman",
        'text.usetex': True,
        "axes.labelsize": 9,
        "axes.titlesize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7
    })
    
    u_array = np.array(u_all)            # (T, 3, N)
    solver_times = np.array(solver_times)
    T, _, N = u_array.shape

    # -----------------------
    # Build discrete input set
    # -----------------------
    discrete_points = []
    for i in range(3):
        for s in (+1, -1):
            v = np.zeros(3); v[i] = s * u_max
            discrete_points.append(v)
    for i in range(3):
        for j in range(i+1, 3):
            for s in (+1, -1):
                v = np.zeros(3); v[i] = s * u_max/2; v[j] = s * u_max/2
                discrete_points.append(v)
    for s in (+1, -1):
        v = np.ones(3) * s * u_max/3
        discrete_points.append(v)
    discrete_points = np.array(discrete_points)  # (P,3)

    # -----------------------
    # Distances per time step
    # -----------------------
    # u_array: shape (T, 3, N)   -> T trials, 3 actuators, N time steps
    # discrete_points: shape (P, 3)

    dist = np.zeros((T, N))
    for t in range(N):
        u_t = u_array[:, :, t]                    # (T,3)
        diffs = u_t[:, None, :] - discrete_points[None, :, :]  
        dists = np.linalg.norm(diffs, axis=2)     # (T,P)
        dist[:, t] = np.min(dists, axis=1)        # closest discrete point per trial

    # -----------------------
    # Metrics based on averaged distance
    # -----------------------
    averaged_distance  = dist.mean(axis=1)       # per-trial averaged distance
    mean_averaged_distance = averaged_distance.mean()  
    median_averaged_distance = np.median(np.median(averaged_distance))
    std_averaged_distance  = averaged_distance.std()
    max_averaged_distance  = averaged_distance.max()  

    # -----------------------
    # Solver time stats
    # -----------------------
    stats = {
        'mean_solver_time': solver_times.mean(),
        'std_solver_time': solver_times.std(),
        'median_solver_time': np.median(solver_times),
        'min_solver_time': solver_times.min(),
        'max_solver_time': solver_times.max(),
    }

    # -----------------------
    # Print summary
    # -----------------------
    print("=== Solver-time summary ===")
    for k, v in stats.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")
    print("")
    print("=== Discreteness summary ===")
    print(f"eps_discrete = {eps_discrete}")
    print(f"Averaged Distance : mean={mean_averaged_distance:.6f}, "
        f"std={std_averaged_distance:.6f}, max={max_averaged_distance:.6f}")

    # -----------------------
    # Plots
    # -----------------------

    # 1) Solver time histogram
    plt.figure(figsize=(3.5, 1.5))
    plt.hist(solver_times, bins=20, color='C1', alpha=0.7)
    plt.xlabel('Solver time [s]')
    plt.ylabel('Count')

    plt.axvline(stats['mean_solver_time'], color='k', ls='--', lw=1.2,
                label=f"Mean={stats['mean_solver_time']:.3f}s")
    plt.axvline(stats['median_solver_time'], color='k', ls=':', lw=1.2,
                label=f"Median={stats['median_solver_time']:.3f}s")

    plt.legend(fontsize=7)
    plt.grid(True)
    if save_plots: 
        plt.savefig("solver_time_hist.pdf", bbox_inches='tight')
    plt.show()

    # 2) Histogram of averaged distance
    plt.figure(figsize=(3.5, 1.5))
    plt.hist(averaged_distance, bins=20, color='C0', alpha=0.8)

    plt.axvline(mean_averaged_distance, color='k', ls='--', lw=1.2,
                label=f"Mean={mean_averaged_distance:.3f}")
    plt.axvline(median_averaged_distance, color='k', ls=':', lw=1.2,
                label=f"Median={median_averaged_distance:.3f}")

    plt.xlabel('$\\bar{d}(u)$')
    plt.ylabel('Count')
    plt.legend(fontsize=7)
    plt.grid(True)
    if save_plots: 
        plt.savefig("hist_avg_dist.pdf", bbox_inches='tight')
    plt.show()


    # -----------------------
    # Results dictionary
    # -----------------------
    results = {
        'averaged_distance': averaged_distance,
        'mean_averaged_distance': mean_averaged_distance,
        'std_averaged_distance': std_averaged_distance,
        'max_averaged_distance': max_averaged_distance,
        'solver_time_stats': stats
    }

    return results


In [ ]:
num_samples = 1000
solver_times, u_all = monte_carlo_cp(num_samples=num_samples, N=400, tf=300.0)

# Summary statistics
print(f"\nMonte Carlo summary ({num_samples} feasible trials):")
print(f"Mean solver time: {np.mean(solver_times):.4f} s | Std: {np.std(solver_times):.4f} s")
print(f"Min solver time: {np.min(solver_times):.4f} s | Max solver time: {np.max(solver_times):.4f} s")

results = discreteness_report(u_all, solver_times, u_max=5/100, save_plots=True)